# PetenFire — Modelo M1 DEFINITIVO: Anual + NDVI + Fire_Lag Features

**27 features** (23 base + 4 historial de fuego)  
**Estrategia:** subsampleo 10:1, sin SMOTE, umbral óptimo desde PR curve

---

## Archivos necesarios en Google Drive (`Mi unidad/petenfire/data/`)

| # | Archivo | Tamaño aprox. | Descripción |
|---|---------|--------------|-------------|
| 1 | `m1_dataset.parquet` | 766 MB | Dataset principal (23 features base + target + split) |
| 2 | `fire_lag_features.parquet` | 46 MB | Historial espacial de fuego (4 features lag) |

---

## Instrucciones previas

1. Sube **ambos** archivos a Google Drive en la ruta: `Mi unidad/petenfire/data/`
2. Activa GPU: `Entorno de ejecución → Cambiar tipo de entorno → GPU T4 (gratis)`
3. Ejecuta **Run All** (`Ctrl+F9`)

---

### Splits temporales
| Split | Años | Filas aprox. |
|-------|------|--------------|
| train | 2018–2022 | 97 M |
| val   | 2023      | 19 M |
| test  | 2024      | 19 M |

### Criterio de éxito
| Métrica | Mínimo |
|---------|--------|
| AUC-PR (val) | > 0.10 |
| F1 (val, umbral óptimo) | > 0.20 |
| best_iteration_ | > 1 |

> **Nota:** El umbral de decisión se determina automáticamente buscando el máximo F1
> en la curva precision-recall de val (Celda 8). El valor se guarda en el artefacto
> y es el que debe usarse en inferencia.

### Historial de experimentos
| Experimento | AUC-ROC | AUC-PR | Decisión |
|-------------|---------|--------|----------|
| Anual base (sin NDVI) | — | 0.004 | Descartado |
| Anual + NDVI | 0.888 | 0.007 | Base de trabajo |
| Seasonal (mar–may) | 0.752 | — | Descartado (best_iter=1) |
| **Anual + NDVI + fire_lag** | **pendiente** | **pendiente** | **Este notebook** |

In [ ]:
# Celda 2 — Instalar dependencias
!pip install -q lightgbm scikit-learn joblib pandas pyarrow numpy psutil
print("Dependencias instaladas.")

In [ ]:
# Celda 3 — Montar Google Drive y definir rutas
import os

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

# ── Rutas de datos ─────────────────────────────────────────────────────────────
DATASET_PATH   = "/content/drive/MyDrive/petenfire/data/m1_dataset.parquet"
FIRE_LAG_PATH  = "/content/drive/MyDrive/petenfire/data/fire_lag_features.parquet"
OUTPUT_DIR     = "/content/drive/MyDrive/petenfire/models/"
MODEL_FILENAME = "m1_lightgbm_final.joblib"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verificar que ambos archivos existen antes de continuar
for path, label in [(DATASET_PATH, "m1_dataset.parquet"), (FIRE_LAG_PATH, "fire_lag_features.parquet")]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró '{label}' en: {path}\n"
            "Asegúrate de haberlo subido a Google Drive en la carpeta "
            "'Mi unidad/petenfire/data/' y de haber montado el Drive correctamente."
        )
    size_mb = os.path.getsize(path) / 1e6
    print(f"[OK] {label} encontrado ({size_mb:.0f} MB): {path}")

print(f"Directorio de salida: {OUTPUT_DIR}")

In [ ]:
# Celda 4 — Cargar datos con join de fire_lag (RAM-eficiente)
#
# Estrategia:
#   1. Cargar fire_lag_features.parquet completo (~46 MB en disco, ~300 MB en RAM).
#   2. Iterar m1_dataset.parquet en batches de 5 M filas.
#   3. Hacer merge LEFT de cada batch con fire_lag por (cell_id, date).
#   4. train: todos los positivos + muestra 10:1 de negativos (seed=42).
#   5. val / test: ESTRATIFICADO — todos los positivos + 20x negativos por batch.
#      (Fix: la muestra aleatoria con prevalencia 0.064% producía ~64 positivos
#       en val → average_precision ruidosa → early stopping en iteración 1.)

import gc

import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq

RANDOM_SEED    = 42
NEG_RATIO      = 10        # negativos por cada positivo en train
VAL_TEST_NEG_RATIO = 20   # negativos por cada positivo en val/test (estratificado)
BATCH_SIZE     = 5_000_000

# ── Features ──────────────────────────────────────────────────────────────────
FIRE_LAG_FEATURE_COLS = [
    "fire_cell_lag30d",
    "fire_cell_lag365d",
    "fire_cell_count_3y",
    "fire_neighbors_30d",
]

FEATURE_COLS = [
    # Clima
    "T2M", "RH2M", "WS10M", "PRECTOTCORR",
    # FWI
    "fwi", "ffmc_val", "dmc_val", "dc_val", "isi_val", "bui_val",
    # Precipitación acumulada
    "prec_acc7d", "prec_acc14d",
    # Vegetación
    "ndvi", "ndvi_lag7", "ndvi_lag14",
    # Topografía
    "elevation_m", "slope_deg", "aspect_deg",
    # Antrópico
    "dist_roads_km", "dist_settlements_km", "is_protected_area",
    # Temporal
    "month", "day_of_year",
    # Historial de fuego (NUEVOS)
    "fire_cell_lag30d", "fire_cell_lag365d", "fire_cell_count_3y", "fire_neighbors_30d",
]

TARGET_COL = "fire_occurred"


def ram_gb() -> float:
    """Retorna la RAM usada por el proceso actual en GB."""
    return psutil.Process().memory_info().rss / 1e9


# ── Paso 1: cargar fire_lag completo ──────────────────────────────────────────
print("Cargando fire_lag_features.parquet...")
fire_lag = pd.read_parquet(
    FIRE_LAG_PATH,
    columns=["cell_id", "date"] + FIRE_LAG_FEATURE_COLS,
)
fire_lag["date"] = pd.to_datetime(fire_lag["date"])
# Convertir a float32 anticipadamente para reducir RAM al hacer merge
for col in FIRE_LAG_FEATURE_COLS:
    fire_lag[col] = fire_lag[col].astype("float32")
print(f"fire_lag cargado: {fire_lag.shape} | RAM: {ram_gb():.2f} GB")

# ── Paso 2: verificar features disponibles en el dataset principal ─────────────
pf = pq.ParquetFile(DATASET_PATH)
schema_names = set(pf.schema_arrow.names)

# Las fire_lag features se añaden via merge; las base deben venir del parquet
BASE_FEATURE_COLS = [c for c in FEATURE_COLS if c not in FIRE_LAG_FEATURE_COLS]
available_base = [c for c in BASE_FEATURE_COLS if c in schema_names]
missing_base   = [c for c in BASE_FEATURE_COLS if c not in schema_names]

if missing_base:
    print(f"AVISO: Features base no encontradas en el parquet (se omitirán): {missing_base}")

# Columnas finales: base disponibles + fire_lag (que vendrán del merge)
available_features = available_base + FIRE_LAG_FEATURE_COLS
BASE_COLS_NEEDED   = ["cell_id", "date", TARGET_COL, "split"] + available_base

print(f"Features base disponibles : {len(available_base)}/{len(BASE_FEATURE_COLS)}")
print(f"Features fire_lag         : {len(FIRE_LAG_FEATURE_COLS)}")
print(f"Total features al modelo  : {len(available_features)}")

# ── Paso 3: iterar dataset en batches y construir splits ──────────────────────
train_pos: list[pd.DataFrame] = []
train_neg: list[pd.DataFrame] = []
val_pos:   list[pd.DataFrame] = []
val_neg:   list[pd.DataFrame] = []
test_pos:  list[pd.DataFrame] = []
test_neg:  list[pd.DataFrame] = []

rng = np.random.default_rng(RANDOM_SEED)

print(f"\nRAM antes de carga del dataset: {ram_gb():.2f} GB")
print(f"Leyendo '{DATASET_PATH}' en batches de {BATCH_SIZE:,} filas...")

for i, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE, columns=BASE_COLS_NEEDED)):
    chunk = batch.to_pandas()
    chunk["date"] = pd.to_datetime(chunk["date"])

    # Join con fire_lag por (cell_id, date)
    chunk = chunk.merge(fire_lag, on=["cell_id", "date"], how="left")

    # Rellenar NaN de fire_lag con 0 (celdas sin historial previo = sin fuego)
    for col in FIRE_LAG_FEATURE_COLS:
        chunk[col] = chunk[col].fillna(0.0).astype("float32")

    # ── train: todos los positivos + muestra NEG_RATIO:1 de negativos ──────
    s_train = chunk[chunk["split"] == "train"]
    if not s_train.empty:
        pos = s_train[s_train[TARGET_COL] == 1]
        neg = s_train[s_train[TARGET_COL] == 0]
        n_neg_keep = min(len(neg), len(pos) * NEG_RATIO)
        train_pos.append(pos)
        if n_neg_keep > 0:
            idx = rng.choice(len(neg), size=n_neg_keep, replace=False)
            train_neg.append(neg.iloc[idx])

    # ── val / test: ESTRATIFICADO — todos los positivos + 20x negativos ────
    # Razón: muestra aleatoria con prevalencia 0.064% ≈ 64 positivos/batch
    # → average_precision ruidosa → early stopping dispara en iteración 1.
    for split_name, pos_store, neg_store in [
        ("val", val_pos, val_neg),
        ("test", test_pos, test_neg),
    ]:
        s = chunk[chunk["split"] == split_name]
        if not s.empty:
            pos = s[s[TARGET_COL] == 1]
            neg = s[s[TARGET_COL] == 0]
            n_neg_keep = min(len(pos) * VAL_TEST_NEG_RATIO, len(neg))
            pos_store.append(pos)
            if n_neg_keep > 0:
                neg_store.append(neg.sample(n_neg_keep, random_state=RANDOM_SEED))

    if (i + 1) % 5 == 0 or i == 0:
        print(f"  Batch {i+1} procesado | RAM: {ram_gb():.2f} GB")

print("Consolidando splits...")
train_df = pd.concat(train_pos + train_neg).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
val_df   = pd.concat(val_pos + val_neg).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_df  = pd.concat(test_pos + test_neg).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

del fire_lag, train_pos, train_neg, val_pos, val_neg, test_pos, test_neg
gc.collect()

print(f"\nTrain : {len(train_df):>10,} filas | {train_df[TARGET_COL].mean()*100:.2f}% positivos")
print(f"Val   : {len(val_df):>10,} filas | {val_df[TARGET_COL].mean()*100:.2f}% positivos")
print(f"Test  : {len(test_df):>10,} filas | {test_df[TARGET_COL].mean()*100:.2f}% positivos")
print(f"\nRAM después de carga: {ram_gb():.2f} GB")

In [ ]:
# Celda 5 — Preparar arrays numpy

import numpy as np

X_train = train_df[available_features].values.astype(np.float32)
y_train = train_df[TARGET_COL].values.astype(np.int32)

X_val = val_df[available_features].values.astype(np.float32)
y_val = val_df[TARGET_COL].values.astype(np.int32)

X_test = test_df[available_features].values.astype(np.float32)
y_test = test_df[TARGET_COL].values.astype(np.int32)

# Liberar DataFrames para recuperar RAM
del train_df, val_df, test_df
gc.collect()

print("Shapes y ratios de positivos:")
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"  {name:6s}: X={X.shape} | y positivos={y.sum():,} ({y.mean()*100:.3f}%)")

# Diagnóstico de NaN por feature
nan_rates = np.isnan(X_train).mean(axis=0)
high_nan = [(available_features[i], nan_rates[i]) for i in range(len(available_features)) if nan_rates[i] > 0.01]
if high_nan:
    print("\nAVISO — Features con >1% NaN en train:")
    for feat, rate in sorted(high_nan, key=lambda x: -x[1]):
        print(f"  {feat:30s} {rate*100:.1f}%")
else:
    print("\n[OK] Ninguna feature con >1% NaN en train.")

print(f"\nRAM después de conversión a numpy: {ram_gb():.2f} GB")

In [ ]:
# Celda 6 — Entrenar LightGBM
#
# Cambios respecto al modelo base:
#   - n_estimators: 500 → 1000 (más árboles disponibles para aprender fire_lag)
#   - num_leaves: default → 63 (mayor capacidad expresiva)
#   - subsample / colsample_bytree: 0.8 (regularización por subsampling)
#   - early_stopping patience: 30 → 50
#   - eval_metric: 'average_precision' → 'auc'
#     (auc es más estable con clases raras; average_precision sigue reportándose
#      en la celda de evaluación — solo el criterio de early stopping usa auc)

import time

import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

LEARNING_RATE = 0.05

model = LGBMClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=LEARNING_RATE,
    scale_pos_weight=1.0,    # subsampleo ya equilibró el dataset
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,           # subsampling de filas por árbol
    colsample_bytree=0.8,    # subsampling de features por árbol
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=-1,
)

# DataFrames con nombres para evitar warning de feature names
X_val_df  = pd.DataFrame(X_val,  columns=available_features)
X_test_df = pd.DataFrame(X_test, columns=available_features)

print(f"Iniciando entrenamiento LightGBM con {len(available_features)} features...")
print(f"Features fire_lag incluidas: {FIRE_LAG_FEATURE_COLS}")
t0 = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(X_val_df, y_val)],
    eval_metric="auc",
    feature_name=available_features,
    callbacks=[
        early_stopping(50, verbose=False),   # paciencia 50 (era 30 en el base)
        log_evaluation(100),
    ],
)

elapsed = time.time() - t0
print(f"\nEntrenamiento completado en {elapsed/60:.1f} min")
print(f"Mejor iteración: {model.best_iteration_}")

if model.best_iteration_ <= 1:
    print("ADVERTENCIA: best_iteration_ = 1. El modelo probablemente no aprendió.")
    print("  → Verifica que el dataset tenga positivos y que las features no sean todas NaN.")

In [ ]:
# Celda 7 — Probabilidades RAW del modelo (sin calibración isotónica)
#
# La calibración isotónica sobre val con prevalencia real (~0.064%) comprime
# todos los scores por debajo de cualquier umbral útil (F1=0 aunque AUC sea alto).
# Solución: usar scores RAW + umbral óptimo hallado en val (Celda 8).

import numpy as np

raw_proba_val  = model.predict_proba(X_val_df)[:, 1]
raw_proba_test = model.predict_proba(X_test_df)[:, 1]

print(f"Probabilidades RAW val  — min={raw_proba_val.min():.6f}, "
      f"max={raw_proba_val.max():.6f}, media={raw_proba_val.mean():.6f}")
print(f"Probabilidades RAW test — min={raw_proba_test.min():.6f}, "
      f"max={raw_proba_test.max():.6f}, media={raw_proba_test.mean():.6f}")
print("\nNota: calibración isotónica deshabilitada (comprimía los scores al rango "
      "de prevalencia real y rompía el F1).")

In [ ]:
# Celda 8 — Búsqueda de umbral óptimo y evaluación
#
# 1. Precision-Recall curve en val → umbral que maximiza F1.
# 2. Se aplica ese mismo umbral (fijado en val) a test. No re-optimizar en test.

import numpy as np
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# ── Paso 1: encontrar umbral óptimo en val ────────────────────────────────────
precisions, recalls, thresholds_pr = precision_recall_curve(y_val, raw_proba_val)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = int(f1_scores[:-1].argmax())
optimal_threshold = float(thresholds_pr[best_idx])
best_f1_val = float(f1_scores[best_idx])

print("Búsqueda de umbral óptimo en val (curva precision-recall):")
print(f"  Umbral óptimo    : {optimal_threshold:.6f}")
print(f"  F1 máximo en val : {best_f1_val:.4f}")
print(f"  Precisión        : {precisions[best_idx]:.4f}")
print(f"  Recall           : {recalls[best_idx]:.4f}")


# ── Paso 2: función de evaluación ─────────────────────────────────────────────
def evaluate_split(
    proba: np.ndarray,
    y: np.ndarray,
    split_name: str,
    threshold: float,
) -> dict:
    """Calcula métricas completas para un split dado un array de probabilidades."""
    pred = (proba >= threshold).astype(int)
    metrics = {
        "auc_roc":   float(roc_auc_score(y, proba)),
        "auc_pr":    float(average_precision_score(y, proba)),
        "f1":        float(f1_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall":    float(recall_score(y, pred, zero_division=0)),
        "threshold": threshold,
    }
    print(f"\n{'='*55}")
    print(f"  {split_name.upper()} | threshold={threshold:.6f}")
    print(f"{'='*55}")
    print(f"  AUC-ROC   : {metrics['auc_roc']:.4f}")
    print(f"  AUC-PR    : {metrics['auc_pr']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    return metrics


# ── Paso 3: evaluar con el umbral óptimo (fijado en val, aplicado a test) ────
val_metrics  = evaluate_split(raw_proba_val,  y_val,  "VAL",  optimal_threshold)
test_metrics = evaluate_split(raw_proba_test, y_test, "TEST", optimal_threshold)

# ── Paso 4: verificar criterios de éxito ──────────────────────────────────────
auc_pr_val = val_metrics["auc_pr"]

print("\n" + "="*55)
print("  CRITERIOS DE ÉXITO")
print("="*55)
print(f"  {'[OK]' if auc_pr_val > 0.10 else '[!!]'} AUC-PR val > 0.10: "
      f"{'PASA' if auc_pr_val > 0.10 else 'FALLA'} ({auc_pr_val:.4f})")
print(f"  {'[OK]' if best_f1_val > 0.20 else '[!!]'} F1 val (umbral óptimo) > 0.20: "
      f"{'PASA' if best_f1_val > 0.20 else 'FALLA'} ({best_f1_val:.4f})")
print(f"  {'[OK]' if model.best_iteration_ > 1 else '[!!]'} best_iteration_ > 1: "
      f"{'PASA' if model.best_iteration_ > 1 else 'FALLA'} ({model.best_iteration_})")
print(f"  [INFO] Umbral óptimo encontrado: {optimal_threshold:.6f}")

all_pass = (auc_pr_val > 0.10) and (best_f1_val > 0.20) and (model.best_iteration_ > 1)
print("="*55)
if all_pass:
    print("  >> MODELO LISTO PARA GUARDAR <<")
else:
    print("  >> REVISA LOS CRITERIOS FALLIDOS ANTES DE REPORTAR <<")
print("="*55)

In [ ]:
# Celda 9 — Guardar modelo
#
# Artefacto en formato dict para facilitar inspección sin reimportar clases.
# threshold = umbral óptimo encontrado en val (no valor fijo).
# calibration = 'none' — se usan probabilidades RAW del LightGBM.

import joblib

artifact = {
    # Objeto del modelo
    "model": model,
    # Configuración de inferencia
    "feature_names": available_features,
    "threshold": optimal_threshold,         # umbral aprendido de val
    "calibration": "none",                  # sin calibración isotónica
    # Métricas de evaluación
    "metrics": {
        "val": val_metrics,
        "test": test_metrics,
    },
    # Metadatos del entrenamiento
    "training_mode": "annual_with_fire_lag",
    "fire_lag_features": FIRE_LAG_FEATURE_COLS,
    "n_features": len(available_features),
    "subsample_ratio": NEG_RATIO,
    "random_seed": RANDOM_SEED,
    "best_iteration": model.best_iteration_,
    "train_split": "2018-2022",
    "val_split": "2023",
    "test_split": "2024",
}

model_path = os.path.join(OUTPUT_DIR, MODEL_FILENAME)
joblib.dump(artifact, model_path, compress=3)   # compress=3 reduce tamaño ~40%

print(f"Modelo guardado en: {model_path}")
print(f"Nombre del archivo : {MODEL_FILENAME}")
print(f"Umbral guardado    : {optimal_threshold:.6f}")
print(f"Calibración        : none (probabilidades RAW)")
print(f"Features           : {len(available_features)} ({len(FIRE_LAG_FEATURE_COLS)} fire_lag)")
print(f"training_mode      : annual_with_fire_lag")

size_mb = os.path.getsize(model_path) / 1e6
print(f"Tamaño del artefacto: {size_mb:.1f} MB")

## Archivos necesarios en Google Drive (`Mi unidad/petenfire/data/`)

1. `m1_dataset.parquet` — dataset principal (766 MB)
2. `fire_lag_features.parquet` — historial espacial de fuego (46 MB)

---

## Después de entrenar

### 1. Descarga el artefacto
Desde Google Drive, descarga `m1_lightgbm_final.joblib` (ruta: `petenfire/models/`).

### 2. Cópialo al proyecto local
```bash
cp ~/Downloads/m1_lightgbm_final.joblib <proyecto>/models_artifacts/m1/
```

### 3. Reporta las métricas

Indica los siguientes valores de la celda 8:

| Métrica | Valor val | Valor test |
|---------|-----------|------------|
| AUC-PR  | _rellenar_ | _rellenar_ |
| AUC-ROC | _rellenar_ | _rellenar_ |
| F1 (umbral óptimo) | _rellenar_ | _rellenar_ |
| Umbral óptimo | _rellenar_ | — |
| best_iteration_ | _rellenar_ | — |

### 4. Si algún criterio falla
- **AUC-PR ≤ 0.10**: ejecuta la celda 5 e inspecciona `np.isnan(X_train).mean(axis=0)` para ver qué features tienen NaN. También verifica que el merge con `fire_lag` no haya generado NaN masivos (si `cell_id` tiene formato distinto entre los dos parquets, el join producirá NaN).
- **F1 ≤ 0.20 (umbral óptimo)**: el modelo no discrimina lo suficiente. Prueba reducir `LEARNING_RATE` a 0.02 o aumentar la paciencia de `early_stopping` a 80.
- **best_iteration_ = 1**: el modelo no aprendió. Verifica `y_train.sum() > 0` y que `X_train` no tenga todas las columnas en NaN.
- **NaN en merge**: comprueba que `cell_id` tenga el mismo formato en ambos parquets (`df['cell_id'].head()`).

### 5. Inferencia con el artefacto guardado
```python
import joblib
import pandas as pd

art = joblib.load("models_artifacts/m1/m1_lightgbm_final.joblib")
model     = art["model"]
threshold = art["threshold"]       # umbral óptimo aprendido de val
features  = art["feature_names"]   # 27 features incluyendo fire_lag

X_new = pd.DataFrame(new_data, columns=features)
proba = model.predict_proba(X_new)[:, 1]
pred  = (proba >= threshold).astype(int)
```

### 6. Comparación con modelo base
| Métrica | Base (anual + NDVI) | Final (+ fire_lag) |
|---------|--------------------|-----------------|
| AUC-ROC | 0.888 | _rellenar_ |
| AUC-PR  | 0.007 | _rellenar_ |
| best_iteration_ | — | _rellenar_ |